# AFTERIMAGE · Earth Engine V2 (notebook propre, à exécuter du haut vers le bas)
Mêmes règles que la V1 : terre = QA bits 0-1 == 2 ; feu = FireMask 7-9 ; observable (O) = FireMask 5 ou feu ;
support commun = observable par Aqua ET VIIRS le même jour. Calcul plus rapide : régions peintes une fois
sur la grille 1 km d'origine, puis une réduction groupée par mois.
- Californie + Colombie-Britannique : QUOTIDIEN, 2002-07-04 -> 2025 (pour Rim)
- Tous les États/provinces : MENSUEL (carte et calendrier raw_only)
- 2020-2021 : dossier AFTERIMAGE_P5_SEALED uniquement

## 1. Connexion et règles (masques)

In [1]:
import ee, os, glob, json, hashlib, calendar, datetime as dt
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ID = 'inlaid-reactor-433400-d6'   # ton projet Earth Engine
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

AQUA  = ee.ImageCollection('MODIS/061/MYD14A1')
VIIRS = ee.ImageCollection('NASA/VIIRS/002/VNP14A1')

COUNTRIES = ['United States of America', 'Canada']
BLOCKS = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.inList('ADM0_NAME', COUNTRIES))
          .select(['ADM0_NAME', 'ADM1_CODE', 'ADM1_NAME']))

# BLOCKS sert uniquement au test de cohérence (ancienne méthode, 2 régions)
BLOCKS = BLOCKS.filter(ee.Filter.inList('ADM1_NAME', ['California', 'British Columbia']))

PROJ = AQUA.first().projection()   # grille sinusoïdale 1 km d'origine : on ne reprojette pas
INCLUDE_COAST = False              # règle figée : terre seule (QA bits 0-1 == 2)

BANDS = ['D_aqua', 'O_aqua', 'D_viirs', 'O_viirs',
         'O_common', 'D_aqua_common', 'D_viirs_common',
         'D_aqua_nh', 'D_viirs_nh']          # _nh = confiance nominale + haute seulement (8-9)

def masks(img):
    """Masques documentés : identiques pour MYD14A1 et VNP14A1 sur ces bits (vérifié catalogue EE)."""
    fm = img.select('FireMask')
    lw = img.select('QA').bitwiseAnd(3)                  # bits 0-1 : 0 eau, 1 côte, 2 terre
    land = lw.eq(2).Or(lw.eq(1)) if INCLUDE_COAST else lw.eq(2)
    fire = fm.gte(7).And(fm.lte(9)).And(land)           # feux 7-9, sur terre uniquement
    fire_nh = fm.gte(8).And(fm.lte(9)).And(land)
    usable = fm.eq(5).Or(fire).And(land)                 # nuage (4), inconnu (6), non traité : exclus
    return fire, fire_nh, usable

def day_image(col, start):
    sub = col.filterDate(start, start.advance(1, 'day'))
    present = sub.size().gt(0)
    empty = ee.Image.constant([0, 0]).rename(['FireMask', 'QA']).toUint32()
    return ee.Image(ee.Algorithms.If(present, sub.first(), empty)), present

def daily_stack(start):
    a, a_ok = day_image(AQUA, start)
    v, v_ok = day_image(VIIRS, start)
    fa, fa_nh, ua = masks(a)
    fv, fv_nh, uv = masks(v)
    common = ua.And(uv)
    img = ee.Image.cat([fa, ua, fv, uv, common, fa.And(common), fv.And(common), fa_nh, fv_nh])
    return img.rename(BANDS).unmask(0).toUint8(), a_ok, v_ok

def daily_table(start):
    img, a_ok, v_ok = daily_stack(start)
    fc = img.reduceRegions(collection=BLOCKS,
                           reducer=ee.Reducer.sum().unweighted(),  # pixel entier, par son centre
                           crs=PROJ, tileScale=4)
    return fc.map(lambda f: f.setGeometry(None).set({
        'date': start.format('YYYY-MM-dd'),
        'aqua_file_present': a_ok, 'viirs_file_present': v_ok}))


Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


## 2. Annuler les anciennes tâches V1 complètes

In [2]:
# Annule les anciennes tâches V1 complètes (afterimage_daily_AAAA) encore en attente ou en cours.
# Les tâches *_quick ne sont PAS touchées.
import re
n = 0
for t in ee.batch.Task.list():
    if re.fullmatch(r'afterimage_daily_\d{4}', t.config['description']) and t.state in ('READY', 'RUNNING'):
        t.cancel(); n += 1
print(n, 'anciennes tâches annulées')


0 anciennes tâches annulées


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


## 3. Définitions V2

In [3]:

ALL_BLOCKS = (ee.FeatureCollection('FAO/GAUL/2015/level1')
              .filter(ee.Filter.inList('ADM0_NAME', ['United States of America', 'Canada']))
              .select(['ADM0_NAME', 'ADM1_CODE', 'ADM1_NAME']))
FOCUS_NAMES = ['California', 'British Columbia']
FOCUS_BLOCKS = ALL_BLOCKS.filter(ee.Filter.inList('ADM1_NAME', FOCUS_NAMES))

def block_image(fc):
    """Identifiant de bloc peint sur la grille sinusoïdale 1 km d'origine (un pixel = un bloc)."""
    return (fc.reduceToImage(['ADM1_CODE'], ee.Reducer.first())
              .rename('block').toInt32().reproject(PROJ))

FOCUS_IMG, ALL_IMG = block_image(FOCUS_BLOCKS), block_image(ALL_BLOCKS)
FOCUS_REGION = FOCUS_BLOCKS.geometry(1000).bounds(1000)
# Domaine complet : un rectangle Amérique du Nord + un rectangle pour les îles Aléoutiennes (au-delà de 180°)
ALL_REGION = ee.Geometry.MultiPolygon(
    [[[[-170, 18], [-50, 18], [-50, 84], [-170, 84], [-170, 18]]],
     [[[172, 50], [180, 50], [180, 56], [172, 56], [172, 50]]]], None, False)

AQUA_START = dt.date(2002, 7, 4)
NB = len(BANDS)

def days_of_month(y, m):
    first = AQUA_START.day if (y, m) == (AQUA_START.year, AQUA_START.month) else 1
    return [dt.date(y, m, d) for d in range(first, calendar.monthrange(y, m)[1] + 1)]

def months_of_year(y):
    return [m for m in range(1, 13) if (y, m) >= (AQUA_START.year, AQUA_START.month)]

def day_img(d):
    return daily_stack(ee.Date(d.isoformat()))[0].toInt32()

def grouped(img, nbands, block_img, region, extra):
    red = ee.Reducer.sum().unweighted().repeat(nbands).group(groupField=nbands, groupName='ADM1_CODE')
    stats = img.addBands(block_img).reduceRegion(reducer=red, geometry=region, crs=PROJ,
                                                 maxPixels=1e13, tileScale=4)
    def to_feat(g):
        g = ee.Dictionary(g)
        sums = ee.List(g.get('sum')).map(lambda x: ee.Number(x).format('%.0f')).join(';')
        return ee.Feature(None, extra).set({'ADM1_CODE': g.get('ADM1_CODE'), 'sums': sums})
    return ee.FeatureCollection(ee.List(stats.get('groups')).map(to_feat))

def focus_daily_year(y):
    """Californie + C.-B. : une réduction groupée par mois, 9 bandes x nb de jours."""
    parts = []
    for m in months_of_year(y):
        days = days_of_month(y, m)
        img = ee.Image.cat([day_img(d).rename([f'{b}_{i}' for b in BANDS]) for i, d in enumerate(days)])
        parts.append(grouped(img, NB * len(days), FOCUS_IMG, FOCUS_REGION,
                             {'first_day': days[0].isoformat(), 'n_days': len(days)}))
    return ee.FeatureCollection(parts).flatten()

def all_monthly_year(y):
    """Tous les blocs : somme mensuelle des 9 comptages (jours sans fichier = 0, comptés à part)."""
    parts = []
    for m in months_of_year(y):
        days = days_of_month(y, m)
        img = ee.ImageCollection.fromImages([day_img(d) for d in days]).sum().rename(BANDS)
        start, end = ee.Date(days[0].isoformat()), ee.Date(days[-1].isoformat()).advance(1, 'day')
        extra = {'month': f'{y}-{m:02d}', 'first_day': days[0].isoformat(), 'n_days': len(days),
                 'aqua_days': AQUA.filterDate(start, end).size(),
                 'viirs_days': VIIRS.filterDate(start, end).size()}
        parts.append(grouped(img, NB, ALL_IMG, ALL_REGION, extra))
    return ee.FeatureCollection(parts).flatten()

def presence_table(y):
    """Fichier présent (1/0) par jour et par capteur, calculé directement (petit, quelques secondes)."""
    days = [d for m in months_of_year(y) for d in days_of_month(y, m)]
    sizes = ee.List([ee.List([AQUA.filterDate(d.isoformat(), (d + dt.timedelta(1)).isoformat()).size(),
                              VIIRS.filterDate(d.isoformat(), (d + dt.timedelta(1)).isoformat()).size()])
                     for d in days]).getInfo()
    return pd.DataFrame({'date': [d.isoformat() for d in days],
                         'aqua_file_present': [int(s[0] > 0) for s in sizes],
                         'viirs_file_present': [int(s[1] > 0) for s in sizes]})

NAMES = pd.DataFrame(ALL_BLOCKS.select(['ADM0_NAME', 'ADM1_CODE', 'ADM1_NAME'])
                     .map(lambda f: f.setGeometry(None)).getInfo()['features'])
NAMES = pd.DataFrame(list(NAMES['properties']))
print('Blocs :', len(NAMES), '| régions focus :', FOCUS_NAMES)


Blocs : 69 | régions focus : ['California', 'British Columbia']


## 4. Test de cohérence V1 / V2 (un jour)

In [4]:
d = dt.date(2017, 7, 15)
v2 = grouped(day_img(d), NB, FOCUS_IMG, FOCUS_REGION, {}).getInfo()['features']
v1 = daily_table(ee.Date(d.isoformat())).filter(ee.Filter.inList('ADM1_NAME', FOCUS_NAMES)).getInfo()['features']
for f in v2:
    code, vals = int(f['properties']['ADM1_CODE']), [int(x) for x in f['properties']['sums'].split(';')]
    ref = next(g['properties'] for g in v1 if int(g['properties']['ADM1_CODE']) == code)
    print(ref['ADM1_NAME'])
    for b, v in zip(BANDS, vals):
        print(f'   {b:15} V1={ref[b]:>9}  V2={v:>9}')
# Attendu : valeurs identiques ou quasi identiques (quelques pixels de bordure peuvent différer).


California
   D_aqua          V1=       40  V2=       42
   O_aqua          V1=   466538  V2=   467915
   D_viirs         V1=       45  V2=       47
   O_viirs         V1=   468462  V2=   469868
   O_common        V1=   465006  V2=   466274
   D_aqua_common   V1=       40  V2=       42
   D_viirs_common  V1=       45  V2=       47
   D_aqua_nh       V1=       38  V2=       40
   D_viirs_nh      V1=       45  V2=       47


## 5. Lancer les exports V2


In [5]:
SEALED = {2020, 2021}
YEARS = list(range(2016, 2025)) + list(range(2002, 2016)) + [2025]
EXISTING = {t.config['description']: t.state for t in ee.batch.Task.list()
            if t.state in ('READY', 'RUNNING', 'COMPLETED')}
for kind, fn in [('focus_daily', focus_daily_year), ('all_monthly', all_monthly_year)]:
    for y in YEARS:
        folder = 'AFTERIMAGE_P5_SEALED' if y in SEALED else 'AFTERIMAGE_EE_v2'
        name = f'afterimage_{kind}_{y}'
        if name in EXISTING:                      # sécurité : ne jamais lancer deux fois
            print(name, 'déjà lancée :', EXISTING[name]); continue
        t = ee.batch.Export.table.toDrive(collection=fn(y), description=name, folder=folder,
                                          fileNamePrefix=name, fileFormat='CSV')
        t.start()
        print(name, '->', folder)

# Tables de présence des fichiers (directement dans le Drive, pas une tâche)
for y in YEARS:
    folder = 'AFTERIMAGE_P5_SEALED' if y in SEALED else 'AFTERIMAGE_EE_v2'
    out = f'/content/drive/MyDrive/{folder}/presence'
    os.makedirs(out, exist_ok=True)
    presence_table(y).to_csv(f'{out}/presence_{y}.csv', index=False)
    print('présence', y, 'OK')


afterimage_focus_daily_2016 déjà lancée : COMPLETED
afterimage_focus_daily_2017 déjà lancée : COMPLETED
afterimage_focus_daily_2018 déjà lancée : COMPLETED
afterimage_focus_daily_2019 déjà lancée : COMPLETED
afterimage_focus_daily_2020 déjà lancée : COMPLETED
afterimage_focus_daily_2021 déjà lancée : COMPLETED
afterimage_focus_daily_2022 déjà lancée : COMPLETED
afterimage_focus_daily_2023 déjà lancée : COMPLETED
afterimage_focus_daily_2024 déjà lancée : COMPLETED
afterimage_focus_daily_2002 déjà lancée : COMPLETED
afterimage_focus_daily_2003 déjà lancée : COMPLETED
afterimage_focus_daily_2004 déjà lancée : COMPLETED
afterimage_focus_daily_2005 déjà lancée : COMPLETED
afterimage_focus_daily_2006 déjà lancée : COMPLETED
afterimage_focus_daily_2007 déjà lancée : COMPLETED
afterimage_focus_daily_2008 déjà lancée : COMPLETED
afterimage_focus_daily_2009 déjà lancée : COMPLETED
afterimage_focus_daily_2010 déjà lancée : COMPLETED
afterimage_focus_daily_2011 déjà lancée : COMPLETED
afterimage_f

## 6. Suivi des tâches (relancer quand tu veux)

In [6]:
# Suivi de TOUTES les tâches du projet (fonctionne même après une déconnexion de Colab)
import collections
states = collections.Counter()
for t in ee.batch.Task.list():
    d = t.config['description']
    if d.startswith('afterimage_'):
        states[t.state] += 1
        if t.state in ('RUNNING', 'FAILED'):
            print(d, t.state, t.status().get('error_message', ''))
print(dict(states))


{'COMPLETED': 48, 'CANCELLED': 28}


## 7. Fonctions de contrôle

In [7]:
AQ = ['D_aqua', 'O_aqua', 'D_aqua_nh']
VI = ['D_viirs', 'O_viirs', 'D_viirs_nh']
CO = ['O_common', 'D_aqua_common', 'D_viirs_common']

def check_year(df, n_blocks):
    """Contrôles du contrat de données ; renvoie (table corrigée, liste de problèmes)."""
    issues = []
    df['date'] = pd.to_datetime(df['date'])
    days = df['date'].nunique()
    expected = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
    if days != len(expected):
        issues.append(f'{len(expected) - days} jours manquants')
    if len(df) != days * n_blocks:
        issues.append(f'lignes {len(df)} != jours x blocs {days * n_blocks}')
    for d, o in [('D_aqua', 'O_aqua'), ('D_viirs', 'O_viirs'), ('D_aqua_common', 'O_common'),
                 ('D_viirs_common', 'O_common'), ('D_aqua_nh', 'D_aqua'), ('D_viirs_nh', 'D_viirs')]:
        bad = (df[d] > df[o]).sum()
        if bad:
            issues.append(f'{bad} lignes avec {d} > {o}')
    bad = (df['O_common'] > df[['O_aqua', 'O_viirs']].min(axis=1)).sum()
    if bad:
        issues.append(f'{bad} lignes avec O_common > min(O_aqua, O_viirs)')
    # fichier absent = indisponible (null), jamais zéro
    df[AQ] = df[AQ].astype('float').where(df['aqua_file_present'] == 1)
    df[VI] = df[VI].astype('float').where(df['viirs_file_present'] == 1)
    both = (df['aqua_file_present'] == 1) & (df['viirs_file_present'] == 1)
    df[CO] = df[CO].astype('float').where(both)
    return df, issues

def sha256(path):
    h = hashlib.sha256(open(path, 'rb').read())
    return h.hexdigest()


## 8. Déplier et contrôler les fichiers V2 (après COMPLETED)

In [8]:
def unpack_focus_daily(csv_path, presence_path):
    df, pres = pd.read_csv(csv_path), pd.read_csv(presence_path)
    rows = []
    for _, r in df.iterrows():
        vals = [float(x) for x in str(r['sums']).split(';')]
        assert len(vals) == NB * int(r['n_days']), 'nombre de valeurs inattendu'
        for i, day in enumerate(pd.date_range(r['first_day'], periods=int(r['n_days']))):
            rec = {'ADM1_CODE': int(r['ADM1_CODE']), 'date': day.strftime('%Y-%m-%d')}
            rec.update({b: vals[i * NB + k] for k, b in enumerate(BANDS)})
            rows.append(rec)
    out = pd.DataFrame(rows).merge(pres, on='date', how='left').merge(NAMES, on='ADM1_CODE', how='left')
    return out[['ADM0_NAME', 'ADM1_CODE', 'ADM1_NAME', 'date', 'aqua_file_present', 'viirs_file_present'] + BANDS]

def unpack_all_monthly(csv_path):
    df = pd.read_csv(csv_path)
    vals = df['sums'].astype(str).str.split(';', expand=True).astype(float)
    vals.columns = BANDS
    out = pd.concat([df[['ADM1_CODE', 'month', 'first_day', 'n_days', 'aqua_days', 'viirs_days']], vals], axis=1)
    out['ADM1_CODE'] = out['ADM1_CODE'].astype(int)
    # mois sans aucun fichier d'un capteur = indisponible ; mois partiel = signalé par *_days < n_days
    out.loc[out['aqua_days'] == 0, ['D_aqua', 'O_aqua', 'D_aqua_nh']] = np.nan
    out.loc[out['viirs_days'] == 0, ['D_viirs', 'O_viirs', 'D_viirs_nh']] = np.nan
    out.loc[(out['aqua_days'] == 0) | (out['viirs_days'] == 0), ['O_common', 'D_aqua_common', 'D_viirs_common']] = np.nan
    return out.merge(NAMES, on='ADM1_CODE', how='left')

V2_DIR = '/content/drive/MyDrive/AFTERIMAGE_EE_v2'
CHECKED = '/content/drive/MyDrive/AFTERIMAGE/ee/checked_v2'
os.makedirs(CHECKED, exist_ok=True)
for p in sorted(glob.glob(f'{V2_DIR}/afterimage_focus_daily_*.csv')):
    y = p.split('_')[-1].split('.')[0]
    df, issues = check_year(unpack_focus_daily(p, f'{V2_DIR}/presence/presence_{y}.csv'), n_blocks=2)
    df.to_parquet(f'{CHECKED}/focus_daily_{y}.parquet', index=False)
    print('focus', y, 'OK' if not issues else issues)
for p in sorted(glob.glob(f'{V2_DIR}/afterimage_all_monthly_*.csv')):
    y = p.split('_')[-1].split('.')[0]
    unpack_all_monthly(p).to_parquet(f'{CHECKED}/all_monthly_{y}.parquet', index=False)
    print('mensuel', y, 'OK')


focus 2002 ['lignes 181 != jours x blocs 362']
focus 2003 ['lignes 365 != jours x blocs 730']
focus 2004 ['lignes 366 != jours x blocs 732']
focus 2005 ['lignes 365 != jours x blocs 730']
focus 2006 ['lignes 365 != jours x blocs 730']
focus 2007 ['lignes 365 != jours x blocs 730']
focus 2008 ['lignes 366 != jours x blocs 732']
focus 2009 ['lignes 365 != jours x blocs 730']
focus 2010 ['lignes 365 != jours x blocs 730']
focus 2011 ['lignes 365 != jours x blocs 730']
focus 2012 ['lignes 366 != jours x blocs 732']
focus 2013 ['lignes 365 != jours x blocs 730']
focus 2014 ['lignes 365 != jours x blocs 730']
focus 2015 ['lignes 365 != jours x blocs 730']
focus 2016 ['lignes 366 != jours x blocs 732']
focus 2017 ['lignes 365 != jours x blocs 730']
focus 2018 ['lignes 365 != jours x blocs 730']
focus 2019 ['lignes 365 != jours x blocs 730']
focus 2022 ['lignes 365 != jours x blocs 730']
focus 2023 ['lignes 365 != jours x blocs 730']
focus 2024 ['lignes 366 != jours x blocs 732']
focus 2025 ['

## 9. Taux R par mois ou par semaine

In [9]:
def rates(df, freq='MS', blocks=None):
    """freq='MS' pour le mois, 'W-MON' pour la semaine ISO (lundi). blocks = liste d'ADM1_CODE à réunir."""
    d = df if blocks is None else df[df['ADM1_CODE'].isin(blocks)]
    cols = AQ + VI + CO
    g = (d.set_index('date')[cols]
           .groupby(pd.Grouper(freq=freq, label='left', closed='left'))
           .sum(min_count=1))                     # tout null -> null, pas 0
    # une période est complète seulement si tous ses jours ont le fichier du capteur
    n_days = d.groupby(pd.Grouper(key='date', freq=freq, label='left', closed='left'))['date'].nunique()
    for s, col in [('aqua', 'O_aqua'), ('viirs', 'O_viirs')]:
        present = d[d[col].notna()].groupby(pd.Grouper(key='date', freq=freq, label='left', closed='left'))['date'].nunique()
        g[f'{s}_days_present'] = present.reindex(g.index).fillna(0).astype(int)
    g['days_in_data'] = n_days.reindex(g.index)
    for s in ['aqua', 'viirs']:
        O = g[f'O_{s}']
        g[f'R_{s}'] = np.where(O > 0, 1000 * g[f'D_{s}'] / O, np.nan)   # O = 0 -> indisponible
    return g

# Exemple (années de développement seulement) :
# dev = pd.concat(pd.read_parquet(p) for p in sorted(glob.glob(f'{CHECKED}/focus_daily_201[6-9].parquet')))
# ca = dev[dev['ADM1_NAME'] == 'California']
# print(rates(ca, 'MS')[['D_aqua', 'O_aqua', 'R_aqua', 'D_viirs', 'O_viirs', 'R_viirs']].head(12))
